In [2]:
import pandas as pd
from pandas import DataFrame

In [5]:
# 1. read csv file
filepath = "../data/Telco-Customer-Churn.csv"
raw_df = pd.read_csv(filepath)
raw_df.head(3)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [10]:
print(raw_df['Contract'].unique())

['Month-to-month' 'One year' 'Two year']


In [ ]:
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv; load_dotenv()

DATABASE_URL=os.get("DATABASE_URL")

engine = create_engine(DATABASE_URL)

# 3. Write the data to SQL
# 'if_exists' can be 'append', 'fail', or 'replace'
# raw_df.to_sql('telco_customer_churn', engine, if_exists='append', index=False)

print("Data imported successfully!")

In [4]:
print(raw_df.columns)

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')


In [11]:
ctg_colms_one_hot = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod']

In [15]:

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(sparse_output=False, dtype=int), ctg_colms_one_hot)
    ], 
    remainder='passthrough'
)

# 2. Fit and transform the DataFrame
# Note: ColumnTransformer returns a numpy array, and changes column order (encoded columns first)
encoded_array = ct.fit_transform(raw_df)

# 3. Rebuild the DataFrame using the transformer's tracking of column names
df_ohe = pd.DataFrame(encoded_array, columns=ct.get_feature_names_out())

# Clean up column names to look nicer (removes the 'onehot__' and 'remainder__' prefixes)
df_ohe.columns = df_ohe.columns.str.replace('onehot__', '').str.replace('remainder__', '')

df_ohe.head(5)

,gender_Female,gender_Male,SeniorCitizen_0,SeniorCitizen_1,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,...,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,customerID,tenure,MonthlyCharges,TotalCharges,Churn
0,1,0,1,0,0,1,1,0,1,0,...,1,0,0,1,0,7590-VHVEG,1,29.85,29.85,No
1,0,1,1,0,1,0,1,0,0,1,...,0,0,0,0,1,5575-GNVDE,34,56.95,1889.5,No
2,0,1,1,0,1,0,1,0,0,1,...,1,0,0,0,1,3668-QPYBK,2,53.85,108.15,Yes
3,0,1,1,0,1,0,1,0,1,0,...,0,1,0,0,0,7795-CFOCW,45,42.3,1840.75,No
4,1,0,1,0,1,0,1,0,0,1,...,1,0,0,1,0,9237-HQITU,2,70.7,151.65,Yes


In [17]:
from sklearn.preprocessing import MinMaxScaler

# 1. Initialize the scaler
scaler = MinMaxScaler()

# 2. Define the numerical columns that need scaling
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

df_ohe['TotalCharges'] = pd.to_numeric(df_ohe['TotalCharges'].replace(" ", 0))

# 3. Fit and transform only the numerical columns
df_ohe[num_cols] = scaler.fit_transform(df_ohe[num_cols])

# Now tenure, MonthlyCharges, and TotalCharges are all between 0 and 1!

In [18]:
df_ohe.head(3)

,gender_Female,gender_Male,SeniorCitizen_0,SeniorCitizen_1,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,...,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,customerID,tenure,MonthlyCharges,TotalCharges,Churn
0,1,0,1,0,0,1,1,0,1,0,...,1,0,0,1,0,7590-VHVEG,0.013889,0.115423,0.003437,No
1,0,1,1,0,1,0,1,0,0,1,...,0,0,0,0,1,5575-GNVDE,0.472222,0.385075,0.217564,No
2,0,1,1,0,1,0,1,0,0,1,...,1,0,0,0,1,3668-QPYBK,0.027778,0.354229,0.012453,Yes
